# Week 13 Lab — Responsible AI + Cybersecurity + Encryption Basics
**ISE 16:540 · AI-Enabled Informatics · Rutgers University · Spring 2026**

---

**How to run:** Click **Runtime → Run all**. Standard library + `cryptography` (installed in setup cell).

| Exercise | What you do |
|---|---|
| 1 — Threat Modeling | Build a structured STRIDE × OWASP threat model programmatically |
| 2 — Cryptography Fundamentals | Implement symmetric and asymmetric encryption; practice key management |
| 3 — M6 Trust & Security Package | Produce your milestone deliverable in structured form |


In [ ]:
# Install cryptography library
!pip install cryptography --quiet

import json, hashlib, hmac, os, datetime, csv, io
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from dataclasses import dataclass, field, asdict
from typing import List, Optional

print("Ready. cryptography library loaded.")
print("Python cryptography — implements NIST-approved algorithms only.")

---
# Exercise 1 — Threat Modeling

Build a structured threat model for an AI system using the STRIDE framework
and cross-referencing the OWASP LLM Top 10.

**STRIDE categories:** S=Spoofing, T=Tampering, R=Repudiation, I=Disclosure, D=Denial, E=Elevation
**Risk score** = likelihood (1–3) × impact (1–3), range 1–9


In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Asset:
    name: str
    description: str
    sensitivity: str          # PUBLIC / INTERNAL / CONFIDENTIAL / RESTRICTED
    impact_if_compromised: str

@dataclass
class Threat:
    id: str
    description: str
    stride: str               # S T R I D E (one or two letters)
    owasp_llm: Optional[str]  # e.g. "LLM01" or None
    affected_asset: str
    attack_vector: str
    likelihood: int           # 1-3
    impact: int               # 1-3

    @property
    def risk_score(self): return self.likelihood * self.impact

    @property
    def severity(self):
        if self.risk_score >= 7: return "HIGH"
        if self.risk_score >= 4: return "MEDIUM"
        return "LOW"

@dataclass
class Control:
    threat_id: str
    control_type: str         # PREVENT / DETECT / RESPOND
    implementation: str
    responsible: str
    verification: str         # how you know it works

print("Threat model classes defined.")

In [ ]:
# ── Build the threat model for the Week 11 procurement agent system ──────
# This is a reference model you can adapt for your own project in Exercise 3.

ASSETS = [
    Asset(
        name="Procurement agent tool access",
        description="Agent can call: query_inventory, calculate_eoq, draft_po, submit_po, delete_records",
        sensitivity="CONFIDENTIAL",
        impact_if_compromised="Unauthorized purchase orders, data deletion, financial loss"
    ),
    Asset(
        name="Supplier database records",
        description="Supplier names, contracts, pricing, contact information",
        sensitivity="CONFIDENTIAL",
        impact_if_compromised="Competitive intelligence exposure, contractual liability"
    ),
    Asset(
        name="Audit log store",
        description="All agent actions, gate decisions, approvals, and denials",
        sensitivity="INTERNAL",
        impact_if_compromised="Loss of accountability evidence, compliance failure"
    ),
    Asset(
        name="LLM API credentials",
        description="API keys for the model provider endpoint",
        sensitivity="RESTRICTED",
        impact_if_compromised="Unauthorized model usage, cost exposure, data leakage"
    ),
]

THREATS = [
    Threat(
        id="T1",
        description="Prompt injection via supplier note field",
        stride="E",  # Elevation of Privilege
        owasp_llm="LLM01",
        affected_asset="Procurement agent tool access",
        attack_vector="Attacker writes adversarial text into supplier notes field; agent retrieves note and follows injected instruction; forwards POs to attacker email",
        likelihood=3, impact=3
    ),
    Threat(
        id="T2",
        description="API key hardcoded in deployment script ends up in version control",
        stride="S",  # Spoofing
        owasp_llm=None,
        affected_asset="LLM API credentials",
        attack_vector="Developer commits .env file to public GitHub repo; attacker scans for exposed keys and makes unauthorized API calls",
        likelihood=2, impact=3
    ),
    Threat(
        id="T3",
        description="Agent submits irreversible purchase order without human approval",
        stride="E",  # Elevation of Privilege
        owasp_llm="LLM08",
        affected_asset="Procurement agent tool access",
        attack_vector="Approval gate misconfigured or bypassed; agent submits $50,000 PO with no human review",
        likelihood=2, impact=3
    ),
    Threat(
        id="T4",
        description="Audit log tampered to hide unauthorized agent actions",
        stride="T",  # Tampering
        owasp_llm=None,
        affected_asset="Audit log store",
        attack_vector="Insider with write access to log database deletes records of unauthorized PO submissions",
        likelihood=1, impact=3
    ),
    Threat(
        id="T5",
        description="Supplier contract data exposed via model context window logging",
        stride="I",  # Information Disclosure
        owasp_llm="LLM06",
        affected_asset="Supplier database records",
        attack_vector="Debug logging captures full context window including retrieved supplier records; logs stored unencrypted",
        likelihood=2, impact=2
    ),
    Threat(
        id="T6",
        description="Runaway agent loop exhausts API budget via recursive tool calls",
        stride="D",  # Denial of Service
        owasp_llm="LLM04",
        affected_asset="LLM API credentials",
        attack_vector="Malformed input causes agent to loop on error; no iteration limit set; $2,000 API bill in 4 hours",
        likelihood=2, impact=2
    ),
]

CONTROLS = [
    Control("T1","PREVENT","Treat all retrieved content as untrusted; sanitize notes before agent context; approval gate for external comms","Security Eng","Red team: inject adversarial supplier note; verify gate blocks action"),
    Control("T2","PREVENT","API keys in secrets manager; pre-commit hook blocks .env commits; 90-day rotation","DevOps","Quarterly secret scanning audit of all repos"),
    Control("T3","PREVENT","Approval gate: submit_po always HIGH; requires explicit human approval; gate test in CI","Engineering","Unit test: submit_po without approval returns DENIED"),
    Control("T4","DETECT","Write-once audit log storage; HMAC signature on each entry; nightly integrity check","Compliance","Alert if any log entry fails HMAC verification"),
    Control("T5","PREVENT","Encrypt audit log AES-256-GCM at rest; PII detection before logging; strict RBAC","Security Eng","Quarterly access review; automated PII scan on log sample"),
    Control("T6","PREVENT","Hard iteration limit=20; cost budget=$10/session; auto-stop and alert on budget exceeded","Engineering","Load test: verify hard stop fires before $10"),
]

print(f"Threat model: {len(ASSETS)} assets, {len(THREATS)} threats, {len(CONTROLS)} controls")

In [ ]:
# Print the threat model report
SEP = "═" * 70
sep = "─" * 70

print(SEP)
print("THREAT MODEL — Procurement Agent System")
print(SEP)

print("\n[ASSETS]")
for a in ASSETS:
    print(f"  {a.name}")
    print(f"    Sensitivity: {a.sensitivity}")
    print(f"    Impact if compromised: {a.impact_if_compromised}")

print(f"\n[THREATS] — sorted by risk score (high → low)")
sorted_threats = sorted(THREATS, key=lambda t: t.risk_score, reverse=True)
for t in sorted_threats:
    ctrl = next((c for c in CONTROLS if c.threat_id == t.id), None)
    print(sep)
    print(f"  {t.id} | STRIDE:{t.stride} | OWASP:{t.owasp_llm or "—"} | "
          f"Score:{t.risk_score} ({t.severity})")
    print(f"  Threat:  {t.description}")
    print(f"  Vector:  {t.attack_vector[:80]}...")
    if ctrl:
        print(f"  Control [{ctrl.control_type}]: {ctrl.implementation[:80]}...")
        print(f"  Verify:  {ctrl.verification}")

print(sep)
print("\n[RISK SUMMARY]")
from collections import Counter
sev_count = Counter(t.severity for t in THREATS)
for s in ["HIGH","MEDIUM","LOW"]:
    print(f"  {s}: {sev_count.get(s,0)} threats")

### YOUR WORK — Exercise 1 Questions

**Q1. Threat T4 (audit log tampering) uses STRIDE category "T" (Tampering). Which additional STRIDE category also applies, and why?**

*Answer:*

---

**Q2. T1 and T3 both have risk score 9 (the maximum). They require different controls — why?**

*Answer:*

---

**Q3. Which threat in this model is NOT directly addressable by technical controls alone, and what additional type of control is required?**

*Answer:*

---
# Exercise 2 — Cryptography Fundamentals

Implement the cryptographic patterns that protect an AI system's data at rest and in transit.
All algorithms used are NIST-approved and implemented via the `cryptography` library.

**No rolling your own crypto.** Use the library's high-level interfaces as demonstrated.


In [ ]:
# ── Part A: Symmetric Encryption (AES-256-GCM) ───────────────────────────
# AES-GCM provides authenticated encryption: confidentiality + integrity in one pass.
# NIST SP 800-57 recommends AES-256 with 96-bit nonce for data at rest.

# Generate a 256-bit (32-byte) AES key
aes_key = AESGCM.generate_key(bit_length=256)
print(f"AES-256 key generated: {len(aes_key)*8} bits")
print(f"Key (hex): {aes_key.hex()[:32]}... [never log this in production]")

# Encrypt a sensitive piece of data (simulate an audit log entry)
aesgcm = AESGCM(aes_key)
nonce = os.urandom(12)   # 96-bit nonce — unique per encryption operation

plaintext = b"""{
  "timestamp": "2026-04-28T14:32:00",
  "session_id": "session-a1b2c3",
  "action": "submit_purchase_order",
  "amount": 2480.00,
  "status": "APPROVED",
  "approver": "ron.iammartino@rutgers.edu"
}"""

ciphertext = aesgcm.encrypt(nonce, plaintext, associated_data=None)
print(f"\nPlaintext length:    {len(plaintext)} bytes")
print(f"Ciphertext length:   {len(ciphertext)} bytes (includes 16-byte auth tag)")
print(f"Nonce (hex):         {nonce.hex()}")

# Decrypt
decrypted = aesgcm.decrypt(nonce, ciphertext, associated_data=None)
print(f"\nDecryption success:  {decrypted == plaintext}")
print("Decrypted content:")
print(decrypted.decode())

In [ ]:
# ── Part B: Asymmetric Encryption (RSA-2048) ─────────────────────────────
# RSA is used for KEY EXCHANGE and DIGITAL SIGNATURES — not bulk data encryption.
# Key exchange: encrypt the AES key with RSA public key so only private key holder can decrypt.

# Generate RSA key pair
rsa_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)
rsa_public_key = rsa_private_key.public_key()

print("RSA-2048 key pair generated.")

# Serialize public key (safe to share)
pub_pem = rsa_public_key.public_bytes(
    serialization.Encoding.PEM,
    serialization.PublicFormat.SubjectPublicKeyInfo
)
print("Public key (first 60 chars):", pub_pem.decode()[:60])

# Envelope encryption: encrypt the AES key with the RSA public key
# This is the NIST SP 800-57 recommended key wrapping pattern
encrypted_aes_key = rsa_public_key.encrypt(
    aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
print(f"\nWrapped AES key length: {len(encrypted_aes_key)} bytes (RSA-encrypted)")
print("The wrapped key can be stored alongside the ciphertext.")
print("Only the RSA private key holder can unwrap it.")

# Unwrap the AES key using the RSA private key
unwrapped_aes_key = rsa_private_key.decrypt(
    encrypted_aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
print(f"\nAES key unwrapped successfully: {unwrapped_aes_key == aes_key}")

In [ ]:
# ── Part C: HMAC Log Signing ─────────────────────────────────────────────
# HMAC (Hash-based Message Authentication Code) provides integrity verification.
# Use it to detect tampering with audit log entries (T4 control from Exercise 1).

import hmac as hmac_lib

# Generate a signing key (separate from the encryption key)
signing_key = os.urandom(32)   # 256-bit HMAC key
print(f"HMAC signing key: {signing_key.hex()[:16]}... (store in secrets manager)")

def sign_log_entry(entry: dict, key: bytes) -> dict:
    """Add an HMAC signature to a log entry to detect tampering."""
    entry_bytes = json.dumps(entry, sort_keys=True).encode()
    signature = hmac_lib.new(key, entry_bytes, "sha256").hexdigest()
    return {**entry, "_hmac": signature}

def verify_log_entry(entry: dict, key: bytes) -> bool:
    """Verify an HMAC-signed log entry. Returns False if tampered."""
    stored_hmac = entry.get("_hmac")
    if not stored_hmac: return False
    entry_clean = {k: v for k, v in entry.items() if k != "_hmac"}
    entry_bytes = json.dumps(entry_clean, sort_keys=True).encode()
    expected = hmac_lib.new(key, entry_bytes, "sha256").hexdigest()
    return hmac_lib.compare_digest(stored_hmac, expected)   # constant-time comparison

# Sign an audit log entry
log_entry = {
    "timestamp": "2026-04-28T14:32:00",
    "step": 7,
    "action": "submit_purchase_order",
    "status": "APPROVED",
    "approver": "ron.iammartino@rutgers.edu"
}
signed_entry = sign_log_entry(log_entry, signing_key)
print(f"\nSigned entry HMAC: {signed_entry['_hmac'][:32]}...")
print(f"Verification (original):  {verify_log_entry(signed_entry, signing_key)}")

# Simulate tampering
tampered = dict(signed_entry)
tampered["status"] = "DENIED"   # attacker changes approval status
print(f"Verification (tampered):  {verify_log_entry(tampered, signing_key)}")
print("\nIf verification returns False, the log has been tampered with.")
print("This is the T4 control from Exercise 1.")

In [ ]:
# ── Part D: Key Lifecycle States (NIST SP 800-57) ────────────────────────
# Demonstrate the key management lifecycle states for an AI system key.

from enum import Enum

class KeyState(Enum):
    PRE_ACTIVATION = "pre_activation"
    ACTIVE         = "active"
    DEACTIVATED    = "deactivated"
    COMPROMISED    = "compromised"
    DESTROYED      = "destroyed"

@dataclass
class ManagedKey:
    key_id: str
    purpose: str          # e.g. "training-data-encryption"
    algorithm: str        # e.g. "AES-256-GCM"
    state: KeyState = KeyState.PRE_ACTIVATION
    created_at: str = field(default_factory=lambda: datetime.datetime.now().isoformat())
    activated_at: str = ""
    rotated_at: str = ""
    state_history: list = field(default_factory=list)

    def transition(self, new_state: KeyState, reason: str = ""):
        old_state = self.state
        self.state = new_state
        event = {
            "timestamp": datetime.datetime.now().isoformat(),
            "from": old_state.value,
            "to": new_state.value,
            "reason": reason
        }
        self.state_history.append(event)
        print(f"  [{self.key_id}] {old_state.value} → {new_state.value}: {reason}")

# Simulate the lifecycle of a training data encryption key
key = ManagedKey(
    key_id="KEY-001",
    purpose="training-data-encryption",
    algorithm="AES-256-GCM"
)
print("Key Lifecycle Simulation — NIST SP 800-57")
print("─" * 55)
key.transition(KeyState.ACTIVE, "Provisioned for training data store encryption")
key.transition(KeyState.DEACTIVATED, "Annual rotation — replaced by KEY-002")
key.transition(KeyState.DESTROYED, "All data re-encrypted with KEY-002; KEY-001 data purged")

print(f"\nFinal state: {key.state.value}")
print(f"State history: {len(key.state_history)} transitions")
for evt in key.state_history:
    print(f"  {evt['from']} → {evt['to']}: {evt['reason']}")

### YOUR WORK — Exercise 2 Questions

**Q1. AES-GCM is described as "authenticated encryption." What does the authentication component protect against, beyond confidentiality?**

*Answer:*

---

**Q2. Why is the RSA-encrypted AES key called "envelope encryption," and why is this pattern recommended by NIST SP 800-57 instead of encrypting data directly with RSA?**

*Answer:*

---

**Q3. The HMAC verification uses `hmac.compare_digest()` instead of `==`. Why is this important for security?**

*Answer:* (Hint: think about timing attacks)

---

**Q4. A key in the DEACTIVATED state should not be used to protect new data, but must not be destroyed yet. Explain why, using the NIST SP 800-57 lifecycle states.**

*Answer:*

---
# Exercise 3 — M6 Trust & Security Package
## This is your graded milestone deliverable

Produce the three-component Trust & Security Package for your course project.
If you do not have your own project data, use the procurement agent system from Exercise 1.

**Required output:** A completed threat model, logging plan, and responsible AI notes
in the structured format below — submitted as your M6 in Canvas.


In [ ]:
# ✏️ YOUR WORK — Define your system assets
# Replace these with the actual assets in your course project system.

YOUR_ASSETS = [
    Asset(
        name="[YOUR ASSET 1 NAME]",
        description="[What is this asset?]",
        sensitivity="CONFIDENTIAL",   # PUBLIC / INTERNAL / CONFIDENTIAL / RESTRICTED
        impact_if_compromised="[What happens if compromised?]"
    ),
    Asset(
        name="[YOUR ASSET 2 NAME]",
        description="[What is this asset?]",
        sensitivity="RESTRICTED",
        impact_if_compromised="[What happens if compromised?]"
    ),
    Asset(
        name="[YOUR ASSET 3 NAME]",
        description="[What is this asset?]",
        sensitivity="INTERNAL",
        impact_if_compromised="[What happens if compromised?]"
    ),
    # Add more assets as needed
]

print(f"Your assets: {len(YOUR_ASSETS)} defined")
for a in YOUR_ASSETS:
    print(f"  [{a.sensitivity}] {a.name}")

In [ ]:
# ✏️ YOUR WORK — Define at least 5 threats
# Use the reference model from Exercise 1 as a starting point.
# Adapt to your specific system context.

YOUR_THREATS = [
    Threat(
        id="YT1",
        description="[Describe the threat]",
        stride="E",         # S T R I D E
        owasp_llm="LLM01",  # or None if no OWASP equivalent
        affected_asset="[Asset name from YOUR_ASSETS]",
        attack_vector="[Specific, system-context-aware attack scenario]",
        likelihood=3,
        impact=3
    ),
    Threat(
        id="YT2",
        description="[Describe the threat]",
        stride="I",
        owasp_llm="LLM06",
        affected_asset="[Asset name]",
        attack_vector="[Specific attack scenario]",
        likelihood=2,
        impact=3
    ),
    Threat(
        id="YT3",
        description="[Describe the threat]",
        stride="T",
        owasp_llm="LLM03",
        affected_asset="[Asset name]",
        attack_vector="[Specific attack scenario]",
        likelihood=1,
        impact=3
    ),
    Threat(
        id="YT4",
        description="[Describe the threat]",
        stride="D",
        owasp_llm="LLM04",
        affected_asset="[Asset name]",
        attack_vector="[Specific attack scenario]",
        likelihood=2,
        impact=2
    ),
    Threat(
        id="YT5",
        description="[Describe the threat]",
        stride="R",
        owasp_llm=None,
        affected_asset="[Asset name]",
        attack_vector="[Specific attack scenario]",
        likelihood=2,
        impact=2
    ),
]

YOUR_CONTROLS = [
    Control("YT1", "PREVENT", "[Implementation]", "[Owner]", "[Verification]"),
    Control("YT2", "DETECT",  "[Implementation]", "[Owner]", "[Verification]"),
    Control("YT3", "PREVENT", "[Implementation]", "[Owner]", "[Verification]"),
    Control("YT4", "PREVENT", "[Implementation]", "[Owner]", "[Verification]"),
    Control("YT5", "DETECT",  "[Implementation]", "[Owner]", "[Verification]"),
]

print(f"Your threat model: {len(YOUR_THREATS)} threats, {len(YOUR_CONTROLS)} controls")
sorted_your = sorted(YOUR_THREATS, key=lambda t: t.risk_score, reverse=True)
for t in sorted_your:
    print(f"  {t.id} [{t.stride}] {t.severity:<7} score={t.risk_score}  {t.description[:55]}")

In [ ]:
# ✏️ YOUR WORK — Logging & Audit Plan
# Define the log events for your system.

LOG_EVENTS = [
    {
        "event_type":    "model_input_output",
        "trigger":       "Every model API call",
        "data_logged":   "Sanitized prompt (PII redacted), response hash, token count",
        "sensitivity":   "INTERNAL",
        "retention_days": 90,
        "linked_threat": "YT2 (Information Disclosure)"
    },
    {
        "event_type":    "gate_decision",
        "trigger":       "Every approval gate evaluation",
        "data_logged":   "Action, risk_level, decision, approver_id, timestamp",
        "sensitivity":   "INTERNAL",
        "retention_days": 365,
        "linked_threat": "YT5 (Repudiation — no audit trail)"
    },
    {
        "event_type":    "auth_event",
        "trigger":       "Login, logout, API key usage",
        "data_logged":   "User ID, timestamp, IP (hashed), action",
        "sensitivity":   "CONFIDENTIAL",
        "retention_days": 365,
        "linked_threat": "YT1 (Spoofing via key theft)"
    },
    {
        "event_type":    "data_access",
        "trigger":       "Any read/write to model weights or training data",
        "data_logged":   "Resource ID, accessor, operation, timestamp",
        "sensitivity":   "INTERNAL",
        "retention_days": 180,
        "linked_threat": "YT3 (Tampering — training data poisoning)"
    },
    {
        "event_type":    "key_usage",
        "trigger":       "Any encryption/decryption with managed key",
        "data_logged":   "Key ID, operation, actor, timestamp (NOT the key material)",
        "sensitivity":   "CONFIDENTIAL",
        "retention_days": 365,
        "linked_threat": "All cryptographic controls (NIST SP 800-57 requirement)"
    },
]

# ✏️ Customize the log events above for your system, then run this output:
print("LOGGING & AUDIT PLAN")
print("═" * 60)
for ev in LOG_EVENTS:
    print(f"\nEvent: {ev['event_type']}")
    print(f"  Trigger:      {ev['trigger']}")
    print(f"  Data logged:  {ev['data_logged']}")
    print(f"  Sensitivity:  {ev['sensitivity']}")
    print(f"  Retention:    {ev['retention_days']} days")
    print(f"  Linked threat: {ev['linked_threat']}")

In [ ]:
# ✏️ YOUR WORK — Responsible AI Notes
# Fill in each section for your specific system.

RESPONSIBLE_AI = {
    "system_name": "[Your AI System Name]",

    "harms_inventory": [
        {
            "harm": "[Describe a specific harm your system could cause]",
            "affected_party": "[Who is harmed: users / third parties / society]",
            "likelihood": "medium",   # low / medium / high
            "severity": "high",
            "mitigation": "[What control reduces this harm]"
        },
        {
            "harm": "[Second harm]",
            "affected_party": "[Who]",
            "likelihood": "low",
            "severity": "medium",
            "mitigation": "[Control]"
        },
        {
            "harm": "[Third harm]",
            "affected_party": "[Who]",
            "likelihood": "medium",
            "severity": "medium",
            "mitigation": "[Control]"
        },
    ],

    "bias_evaluation_plan": {
        "subgroups": "[Which user or demographic subgroups does your system affect?]",
        "metric": "[Which fairness metric — demographic parity / equalized odds / equal opportunity]",
        "threshold": "[What performance gap between groups is acceptable? e.g. <5%]",
        "nist_function": "Measure",  # which NIST AI RMF function this maps to
        "measurement_frequency": "[How often you re-evaluate — e.g. quarterly]"
    },

    "transparency_disclosure": (
        "[Write one paragraph suitable for user-facing documentation."
         " Describe: what the system does, what it cannot do, what data it uses, "
         " and how users can contest a decision if needed.]"
    ),

    "human_oversight": [
        "[Decision type that requires human review]: [Trigger condition]",
        "[Second decision type]: [Trigger condition]",
    ]
}

# Print the responsible AI notes
print("RESPONSIBLE AI NOTES")
print("═" * 60)
print(f"System: {RESPONSIBLE_AI['system_name']}")

print("\n[HARMS INVENTORY]")
for i, h in enumerate(RESPONSIBLE_AI["harms_inventory"], 1):
    print(f"  Harm {i}: {h['harm']}")
    print(f"    Affects: {h['affected_party']} | Likelihood: {h['likelihood']} | Severity: {h['severity']}")
    print(f"    Mitigation: {h['mitigation']}")

bep = RESPONSIBLE_AI["bias_evaluation_plan"]
print(f"\n[BIAS EVALUATION PLAN]")
print(f"  Subgroups:   {bep['subgroups']}")
print(f"  Metric:      {bep['metric']}")
print(f"  Threshold:   {bep['threshold']}")
print(f"  NIST RMF:    {bep['nist_function']}")
print(f"  Frequency:   {bep['measurement_frequency']}")

print(f"\n[TRANSPARENCY DISCLOSURE]")
print(f"  {RESPONSIBLE_AI['transparency_disclosure']}")

print(f"\n[HUMAN OVERSIGHT]")
for o in RESPONSIBLE_AI["human_oversight"]:
    print(f"  • {o}")

In [ ]:
# Verification — all checks must pass before submitting

checks = [
    ("Assets: >= 3 defined",            len(YOUR_ASSETS) >= 3),
    ("Threats: >= 5 defined",           len(YOUR_THREATS) >= 5),
    ("Controls: one per threat",        len(YOUR_CONTROLS) >= len(YOUR_THREATS)),
    ("OWASP referenced",               any(t.owasp_llm for t in YOUR_THREATS)),
    ("Logging plan: >= 5 events",       len(LOG_EVENTS) >= 5),
    ("Responsible AI: 3 harms",         len(RESPONSIBLE_AI["harms_inventory"]) >= 3),
    ("Transparency disclosure written", len(RESPONSIBLE_AI["transparency_disclosure"]) > 80),
    ("Human oversight specified",       len(RESPONSIBLE_AI["human_oversight"]) >= 1),
]

all_ok = True
for label, ok in checks:
    print(f"  {chr(9989) if ok else chr(10060)} {label}")
    if not ok: all_ok = False
print()
print("Ready to submit M6." if all_ok else "Complete the failing checks before submitting.")

---
## M6 Submission Checklist

- [ ] **Runtime → Restart and run all** — every cell runs without errors
- [ ] Exercise 1: all three questions answered
- [ ] Exercise 2: all four questions answered; all encryption demos run successfully
- [ ] Exercise 3: verification cell shows all eight checks passing
- [ ] Exercise 3: threat model has system-specific attack vectors (not generic descriptions)
- [ ] Exercise 3: OWASP LLM Top 10 referenced for at least 3 threats
- [ ] Exercise 3: NIST SP 800-57 referenced in at least one control
- [ ] Exercise 3: transparency disclosure is a complete paragraph, not a placeholder
- [ ] Submitted to Canvas → Assignments → M6

---
*ISE 16:540 · AI-Enabled Informatics · Week 13 Lab · Spring 2026 · Rutgers University*